# Hockey-Reference Scrape, Cleaning, and Final Dataset

Repo: `ewnike/NHL-Beyond-27` — *MADS Milestone I Project*  
Python: 3.13.7 (pyenv env: `nhl_beyond27-3.13.7`)  
Editors/Tools: VSCode, Git/GitHub, Postgres + pgAdmin  
Logs: written via `log_utils.py` (default `logs/`)

**This notebook covers:**
1) What was scraped from Hockey-Reference (Standard + EV) and why
2) Normalization and TOI parsing rules (avg vs. total)
3) (Optional) Running the scrapers to reproduce raw CSVs
4) Building normalized outputs and the merged final table
5) Curating the final (drop goalies, compute EV minutes, trim columns)
6) (Optional) Cap-hit merge
7) (Optional) Load curated data into Postgres
8) Validation/diffs and lessons learned

## Scrape scope

- Seasons: 2013–14 → 2024–25  
- Strength: EV/5v5 focus (plus “Standard” season pages)  
- Sources (per season):
  - Standard skater pages (all situations)
  - Even-strength time-on-ice pages

## Normalization rules

- Column headers → snake_case (e.g., `CF% Rel` → `cf_rel`)  
- `player`: lowercase, trim, collapse whitespace  
- `tm`: uppercase; recognize `TOT`, `2TM`, `3TM`, etc.  
- TOI parsing understands:
  - Per-game `mm:ss` (multiply by GP)
  - Season total `mmmm:ss`
  - Rare `HH:MM:SS`  
- Use `time_utils.compute_total_toi` and `time_utils.seconds_to_hms`.

## Multi-team seasons & merge policy

- Standard:
  - If an `nTM` row exists (e.g., `2TM`), keep that as `tm='TOT'` and drop per-team rows.
  - If no `nTM`, synthesize a `TOT` row by summing numerics (mode position for `pos`).
- Even strength:
  - Do **not** collapse by team in files; compute per-game → season totals.
  - For the merge only, aggregate EV by `(player, season)` to avoid row multiplication (sum TOI seconds; TOI-weighted CF% if available).

## Order of operations

- `scrap_hockey_ref_player.py` — scrape Standard pages per season → `data/seasons/`  
- `scrap_hcky_ref_evenstrength.py` — scrape EV pages per season → `data/even_strength/`  
- **(Alternative to scraping)** `download_ref_hockey.py` — pull pre-scraped Standard + EV CSVs from S3 into `data/seasons/` and `data/even_strength/`  
- `build_ref_hockey.py` — normalize, fix Standard nTM/TOT, compute EV totals, merge → `data/outputs/`  
- `drop_goalies_etal_inplace.py` — drop goalies/zero-TOI, add EV minutes, drop `cf_rel`/`pos`, save final  
- *(Optional)* cap-hit merge → `data/outputs/hockeyref_final_with_cap.csv`  
- *(Optional)* DB load via `build_ref_hockey_data_table.py`


**Outputs**
- `data/outputs/hockeyref_std_concat.csv`  
- `data/outputs/hockeyref_even_concat.csv`  
- `data/outputs/hockeyref_final.csv` (after curation)


**Run this to set paths, helpers...**

In [1]:
# UNIVERSAL BOOTSTRAP
import platform
import re
import sys
import warnings
from datetime import UTC
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

CWD = Path.cwd()
ROOT = CWD if (CWD / "src").exists() and (CWD / "data").exists() else CWD.parent
assert (ROOT / "src").exists() and (ROOT / "data").exists(), "Run from project root or notebooks/."

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ASSETS_DIR = ROOT / "data" / "assets"
OUTPUTS_DIR = ROOT / "data" / "outputs"
FIGS_DIR = OUTPUTS_DIR / "figs"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)


def _safe(s):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s)).strip("_")  # noqa: E702


def list_assets():
    return sorted(
        {p.stem for p in ASSETS_DIR.glob("*.csv")} | {p.stem for p in ASSETS_DIR.glob("*.csv.gz")}
    )


def load_asset(stem: str):
    for ext in (".csv", ".csv.gz"):
        p = ASSETS_DIR / f"{stem}{ext}"
        if p.exists():
            df = pd.read_csv(p, compression="infer")
            if "rel_age" in df.columns:
                df["rel_age"] = pd.to_numeric(df["rel_age"], errors="coerce")
            return df
    raise FileNotFoundError(f"Missing {stem}.csv in {ASSETS_DIR}")


def require_cols(df, cols, name="df"):
    miss = set(cols) - set(df.columns)
    assert not miss, f"{name} missing columns: {sorted(miss)}"


def save_output_csv(df, name, *, versioned=False, index=False, float_format=None):
    from datetime import datetime

    stem = _safe(name) + (f"-{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}" if versioned else "")
    path = OUTPUTS_DIR / f"{stem}.csv"
    df.to_csv(path, index=index, float_format=float_format)
    print(f"[outputs] {path.name}  rows={len(df):,}  cols={df.shape[1]}")
    return str(path)


def save_fig_matplotlib(fig, name, dpi=180):
    path = FIGS_DIR / f"{_safe(name)}.png"
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"[fig] {path.name}")
    return str(path)


print(f"Python {platform.python_version()}  |  Pandas {pd.__version__}  |  Numpy {np.__version__}")
print("Assets:", ASSETS_DIR.resolve())
print("Outputs:", OUTPUTS_DIR.resolve())
print("Available asset tables:", list_assets())

Python 3.13.7  |  Pandas 2.3.2  |  Numpy 2.3.3
Assets: /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/assets
Outputs: /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/outputs
Available asset tables: ['peak_player_season_stats', 'player_five_year_aligned', 'player_five_year_aligned_cap', 'player_five_year_aligned_z', 'player_five_year_aligned_z_cohort', 'player_hockeyref_even_toi', 'player_hockeyref_even_toi_sample_200', 'player_peak_season', 'player_peak_season_one_row', 'player_streak_seasons']


**Load Necessary Assets...**

In [2]:
# === Load the correct assets ===
df_z = load_asset("player_five_year_aligned_z")
df_z_cohort = load_asset("player_five_year_aligned_z_cohort")

# --- Normalize column names we rely on
for df in (df_z, df_z_cohort):
    # sometimes 'pos' instead of 'position'
    if "position" not in df.columns and "pos" in df.columns:
        df.rename(columns={"pos": "position"}, inplace=True)

    # derive role if missing (D vs F) from position
    if "role" not in df.columns:
        if "position" in df.columns:
            pos0 = df["position"].astype(str).str.upper().str[0]
            df["role"] = np.where(pos0.eq("D"), "D", "F")
        else:
            raise KeyError("Neither 'role' nor 'position' exists to infer role.")

# --- Required columns for downstream
need = {"player", "position", "role", "season", "peak_year", "rel_age"}
require_cols(df_z, need, "df_z")
require_cols(df_z_cohort, need, "df_z_cohort")

# --- Ensure rel_age is numeric and in expected set
VALID_RELS = {-2, -1, 0, 1, 2}
df_z["rel_age"] = pd.to_numeric(df_z["rel_age"], errors="coerce")
df_z_cohort["rel_age"] = pd.to_numeric(df_z_cohort["rel_age"], errors="coerce")

print("[ok] core assets loaded")
print("df_z    shape:", df_z.shape, "| roles:", df_z["role"].value_counts(dropna=False).to_dict())
print(
    "df_coh  shape:",
    df_z_cohort.shape,
    "| roles:",
    df_z_cohort["role"].value_counts(dropna=False).to_dict(),
)

[ok] core assets loaded
df_z    shape: (1410, 22) | roles: {'F': 900, 'D': 510}
df_coh  shape: (1410, 16) | roles: {'F': 900, 'D': 510}


**Cache canonical copies to data/outputs**

In [3]:
# === Load the correct assets ===
df_z = load_asset("player_five_year_aligned_z")
df_z_cohort = load_asset("player_five_year_aligned_z_cohort")

# --- Normalize column names we rely on
for df in (df_z, df_z_cohort):
    # sometimes 'pos' instead of 'position'
    if "position" not in df.columns and "pos" in df.columns:
        df.rename(columns={"pos": "position"}, inplace=True)

    # derive role if missing (D vs F) from position
    if "role" not in df.columns:
        if "position" in df.columns:
            pos0 = df["position"].astype(str).str.upper().str[0]
            df["role"] = np.where(pos0.eq("D"), "D", "F")
        else:
            raise KeyError("Neither 'role' nor 'position' exists to infer role.")

# --- Required columns for downstream
need = {"player", "position", "role", "season", "peak_year", "rel_age"}
require_cols(df_z, need, "df_z")
require_cols(df_z_cohort, need, "df_z_cohort")

# --- Ensure rel_age is numeric and in expected set
VALID_RELS = {-2, -1, 0, 1, 2}
df_z["rel_age"] = pd.to_numeric(df_z["rel_age"], errors="coerce")
df_z_cohort["rel_age"] = pd.to_numeric(df_z_cohort["rel_age"], errors="coerce")

print("[ok] core assets loaded")
print("df_z    shape:", df_z.shape, "| roles:", df_z["role"].value_counts(dropna=False).to_dict())
print(
    "df_coh  shape:",
    df_z_cohort.shape,
    "| roles:",
    df_z_cohort["role"].value_counts(dropna=False).to_dict(),
)

[ok] core assets loaded
df_z    shape: (1410, 22) | roles: {'F': 900, 'D': 510}
df_coh  shape: (1410, 16) | roles: {'F': 900, 'D': 510}


**Smoke Test**

In [4]:
def smoke(df, name):
    print(f"[smoke] {name}: rows={len(df):,}, cols={df.shape[1]}")


smoke(df_z, "df_z")
smoke(df_z_cohort, "df_z_cohort")
print("[ok] Preflight checks passed")

[smoke] df_z: rows=1,410, cols=22
[smoke] df_z_cohort: rows=1,410, cols=16
[ok] Preflight checks passed


**Imports**

In [5]:
# Minimal viz imports (put near the top of the notebook)
import numpy as np
import pandas as pd

In [6]:
# import logging
# from pathlib import Path

# # --- Dirs ---
# DATA = Path("data"); OUT = DATA / "outputs"
# DATA.mkdir(exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)

# print("Working dir:", Path.cwd())
# print("Data dir   :", DATA.resolve())
# print("Outputs dir:", OUT.resolve())

# # --- Quiet, safe .env loading ---
# try:
#     # hush python-dotenv's "could not parse" noise
#     logging.getLogger("dotenv.main").setLevel(logging.ERROR)

#     from dotenv import dotenv_values, find_dotenv, load_dotenv

#     env_path = find_dotenv(usecwd=True)
#     if env_path and Path(env_path).is_file():
#         # pre-parse silently (doesn't spam logs); helps catch truly broken encodings
#         _ = dotenv_values(env_path)  # returns dict; won't print parser warnings
#         load_dotenv(env_path)        # actually load into os.environ, quietly
#         print("Loaded .env from:", env_path)
#     else:
#         print(".env not found (skipping)")
# except ImportError:
#     # python-dotenv not installed—fine, just continue
#     print("python-dotenv not installed; skipping .env load")
# except Exception as e:
#     # final safety: don't crash the notebook over a noisy .env
#     print(f"Skipped .env load due to: {e.__class__.__name__}: {e}")



## Verify the 3 CSVs exist + quick previews (CSV-only, no scrape)

In [7]:
import glob
from pathlib import Path

import pandas as pd

DATA = Path("data")
OUT = DATA / "outputs"
DATA.mkdir(exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Adjust these patterns to your actual filenames if needed
patterns = {
    "skaters_standard": "data/seasons/*.csv",  # e.g., per-season standard skater files or one merged file
    "skaters_evenstrength": "data/even_strength/*.csv",  # ES skater files
    "goalies": "data/goalies/*.csv",  # goalie files
}

found = {k: sorted(glob.glob(pat)) for k, pat in patterns.items()}
for k, files in found.items():
    print(f"{k}: {len(files)} file(s)")
    for f in files[:3]:
        print("  -", f)
    if len(files) > 3:
        print("  - ...")


# Try to read one representative file from each bucket (or a merged CSV if that's your setup)
def _first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return p
    return None


samples = {
    "skaters_standard": _first_existing(found["skaters_standard"])
    or (OUT / "ref_skaters_standard.csv"),
    "skaters_evenstrength": _first_existing(found["skaters_evenstrength"])
    or (OUT / "ref_skaters_evenstrength.csv"),
    "goalies": _first_existing(found["goalies"]) or (OUT / "ref_goalies_standard.csv"),
}

for label, path in samples.items():
    p = Path(path)
    if not p.exists():
        print(f"[skip] No sample for {label}: expected at {p}")
        continue
    df = pd.read_csv(p)
    print(f"\n{label} → {p.resolve()}  shape={df.shape}")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(df.head(10).to_string(index=False))

skaters_standard: 12 file(s)
  - data/seasons/nhl_player_seasons_2014.csv
  - data/seasons/nhl_player_seasons_2015.csv
  - data/seasons/nhl_player_seasons_2016.csv
  - ...
skaters_evenstrength: 12 file(s)
  - data/even_strength/nhl_player_even_toi_2014.csv
  - data/even_strength/nhl_player_even_toi_2015.csv
  - data/even_strength/nhl_player_even_toi_2016.csv
  - ...
goalies: 1 file(s)
  - data/goalies/eh_goalies_study_years.csv

skaters_standard → /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/seasons/nhl_player_seasons_2014.csv  shape=(1126, 6)
 season_end            player  age   gp      toi teams
       2014     Sidney Crosby 26.0 80.0 29:17:47   PIT
       2014      Ryan Getzlaf 28.0 77.0 27:18:58   ANA
       2014     Claude Giroux 26.0 82.0 27:55:38   PHI
       2014      Tyler Seguin 22.0 80.0 25:47:49   DAL
       2014       Corey Perry 28.0 81.0 26:17:59   ANA
       2014       Taylor Hall 22.0 75.0 25:01:02   EDM
       2014       Phil Kessel 26.0 82.0 28:14:37   TOR

## Now, run build_ref_hockey. 
**The files in folders seasons and even_strength get concatenated. Three csv files get output to the outputs folder. The file in hockeyref_final.csv on first pass is the first reduction of the standard file.**

In [8]:
# Run build_ref_hockey.py and preview outputs (head 10)
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
OUT = ROOT / "data" / "outputs"
SCRIPT_CANDIDATES = [ROOT / "build_ref_hockey.py", ROOT / "scripts" / "build_ref_hockey.py"]

script = next((p for p in SCRIPT_CANDIDATES if p.exists()), None)
assert script is not None, f"build_ref_hockey.py not found at: {SCRIPT_CANDIDATES}"

# Run with your defaults; adjust args here only if you want filters
cmd = [sys.executable, str(script)]
print("[run ]", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    print(res.stdout)
    print(res.stderr)
    raise RuntimeError("build_ref_hockey.py failed")

# Locate expected outputs
std_csv = OUT / "hockeyref_std_concat.csv"
even_csv = OUT / "hockeyref_even_concat.csv"
final_csv = OUT / "hockeyref_final.csv"

missing = [p for p in [std_csv, even_csv, final_csv] if not p.exists()]
assert not missing, f"Missing outputs: {', '.join(str(m) for m in missing)}"

# Print shapes + head(10) of the final table only (keep it tight)
print(f"[ ok ] Wrote:\n  {std_csv}\n  {even_csv}\n  {final_csv}")
df_final = pd.read_csv(final_csv)
print("final shape:", df_final.shape)
with pd.option_context("display.max_columns", None, "display.width", 160):
    print("\nhockeyref_final.csv (head 10):")
    print(df_final.head(10).to_string(index=False))

[run ] /Users/ericwiniecke/.pyenv/versions/nhl_beyond27-3.13.7/bin/python /Users/ericwiniecke/Documents/github/NHL-Beyond-27/build_ref_hockey.py
[ ok ] Wrote:
  /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/outputs/hockeyref_std_concat.csv
  /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/outputs/hockeyref_even_concat.csv
  /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/outputs/hockeyref_final.csv
final shape: (12114, 11)

hockeyref_final.csv (head 10):
        player season   gp  age  tm pos  toi_seconds_total_std toi_total_hms_std  toi_seconds_total_ev  cf_rel toi_total_hms_ev
    aaron ness  13-14 20.0 23.0 NYI NaN                  17753           4:55:53                 825.0    -1.3          0:13:45
aaron palushaj  13-14  2.0 24.0 CAR NaN                   1119           0:18:39                 525.0    12.8          0:08:45
    aaron rome  13-14 25.0 30.0 DAL NaN                  19632           5:27:12                 722.0    -5.3          0:12:02
aar

## Run drop_goalies_etal_inplace to remove extra players and continue formatting the data. The magic number needed for both even strength and all regular season players to match is 10939!

In [9]:
# run your script exactly as-is
!python drop_goalies_etal_inplace.py

# quick peek so the graders see the result
import pandas as pd

df = pd.read_csv("data/outputs/hockeyref_final.csv")
print(df.head(10).to_string(index=False))

2025-10-03 15:35:15,452 - INFO - __main__ - Dropped columns: cf_rel, pos
2025-10-03 15:35:15,470 - INFO - __main__ - Final rows before:                 12114
2025-10-03 15:35:15,470 - INFO - __main__ - Removed (goalie matches):          1035
2025-10-03 15:35:15,470 - INFO - __main__ - Removed (zero/NaN toi_seconds_total_ev): 140
2025-10-03 15:35:15,470 - INFO - __main__ - Removed total:                     1175
2025-10-03 15:35:15,470 - INFO - __main__ - Final rows after:                  10939
2025-10-03 15:35:15,471 - INFO - __main__ - Sample EV minutes (2-dec): [275.0, 17.5, 300.83]
        player season   gp  age  tm  toi_seconds_total_std toi_total_hms_std  toi_seconds_total_ev toi_total_hms_ev  toi_even_strength_min  toi_even_strength_min_str
    aaron ness  13-14 20.0 23.0 NYI                  17753           4:55:53                   825          0:13:45                 275.00                     275.00
aaron palushaj  13-14  2.0 24.0 CAR                   1119           0:18:3

## Finally, run build_ref_hockey_data_table.py and create your data table. Now you can use the cleaned Hockey Reference data to verify the validity of the data from Evolving Hockey.
**(Note: we already have the final data in a csv format which we will render a sample of for the readers viewing pleasure.)**

In [10]:
# Notebook-safe: no DB, just write the full table + a sample
from pathlib import Path

import pandas as pd

OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)

src = OUT / "hockeyref_final.csv"  # input your script already produced
full_out = OUT / "player_hockeyref_even_toi.csv"  # "final table" for readers
sample_out = OUT / "player_hockeyref_even_toi_sample_200.csv"

assert src.exists(), f"Missing {src}"
df = pd.read_csv(src)

# write full + sample
df.to_csv(full_out, index=False)
df.head(200).to_csv(sample_out, index=False)

print("Wrote:", full_out.resolve(), "shape=", df.shape)
print("Wrote:", sample_out.resolve(), "shape=", (min(len(df), 200), df.shape[1]))
with pd.option_context("display.max_columns", None, "display.width", 160):
    print("\nplayer_hockeyref_even_toi (head 10):")
    print(df.head(10).to_string(index=False))

Wrote: /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/outputs/player_hockeyref_even_toi.csv shape= (10939, 11)
Wrote: /Users/ericwiniecke/Documents/github/NHL-Beyond-27/data/outputs/player_hockeyref_even_toi_sample_200.csv shape= (200, 11)

player_hockeyref_even_toi (head 10):
        player season   gp  age  tm  toi_seconds_total_std toi_total_hms_std  toi_seconds_total_ev toi_total_hms_ev  toi_even_strength_min  toi_even_strength_min_str
    aaron ness  13-14 20.0 23.0 NYI                  17753           4:55:53                   825          0:13:45                 275.00                     275.00
aaron palushaj  13-14  2.0 24.0 CAR                   1119           0:18:39                   525          0:08:45                  17.50                      17.50
    aaron rome  13-14 25.0 30.0 DAL                  19632           5:27:12                   722          0:12:02                 300.83                     300.83
aaron volpatti  13-14 41.0 28.0 WSH              

## Validation & diffs

- Use `diff_players_by_season.py` to find mismatches across sources/seasons.  
- Confirm EV totals look plausible after `mm:ss × GP` conversion (spot check extremes).


In [11]:
# Example (adjust to your script’s CLI)
# !python diff_players_by_season.py \
#   --std data/outputs/hockeyref_std_concat.csv \
#   --even data/outputs/hockeyref_even_concat.csv \
#   --out data/outputs/_diagnostics


## Lessons learned

- Avg vs. total TOI: multiply only when parsing `mm:ss` per-game + GP.  
- Multi-team seasons: use `nTM` if present; otherwise synthesize consistent `TOT`.  
- Merge hygiene: normalize names/teams/seasons before grouping/joins.  
- Diagnostics early: write small samples to `_diagnostics/`.  
- Keep EV aggregation minimal (only for `(player, season)` merge).


## Appendix: time parsing & helpers

- Regex patterns:
  - `_mmss_re` → per-game `mm:ss` (0–59 min)
  - `_mmmmss_re` → season totals `mmmm:ss`
  - `_hhmmss_re` → `HH:MM:SS` edge cases
- Functions (`time_utils.py`):
  - `compute_total_toi(df, toi_col="toi")` → adds `toi_seconds_total`, `toi_total_hms`
  - `seconds_to_hms(x)` → friendly `H:MM:SS`


**Final note.** Scraping Hockey-Reference was time-consuming, but worthwhile: we built clean season and EV files across 12 years and used them for an independent, back-of-the-envelope verification of **player counts** against Evolving-Hockey. The HR data wasn’t used for modeling in Book 2—its role was strictly **coverage validation** to confirm we were analyzing the right population.
